# Cierre mensual de saldos

Consolidacion del saldo de cierre por producto hacia la capa gold.

## 1. Cabecera
> **Descripcion:** Informacion general del proceso: objetivo, version, responsable y tablas involucradas.

In [ ]:
# -------------------------------------------------------------------------
# PROYECTO       : Finanzas - Saldos
# PROCESO        : ETL_CIERRE_SALDOS
# OBJETIVO       : Consolidar el saldo de cierre mensual por producto
# VERSION        : 1.0.0
# DESARROLLADOR  : Eduardo Fajardo
# FECHA          : 28/08/2026
# TABLA FUENTE   : mb_silver_prod.fin.h_saldo
# TABLA DESTINO  : mb_gold_prod.finanzas.fct_saldo_cierre
# FRECUENCIA     : Diaria
# -------------------------------------------------------------------------

## 2. Importacion de librerias
> **Descripcion:** Librerias estandar, de terceros y locales, en ese orden.

In [ ]:
import logging
import time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 3. Lectura de parametros
> **Descripcion:** Parametros de ejecucion recibidos por widgets, para que el mismo codigo corra en cualquier ambiente.

In [ ]:
dbutils.widgets.text("p_periodo", "")
dbutils.widgets.text("p_catalogo", "")

var_fecha_proceso = dbutils.widgets.get("p_periodo")
var_catalogo = dbutils.widgets.get("p_catalogo")

logger = logging.getLogger("ETL_CIERRE_SALDOS")
logger.setLevel(logging.INFO)

ini_proceso = time.perf_counter()
logger.info("Inicio del proceso ETL_CIERRE_SALDOS")
logger.info("Parametros recibidos por widgets: fecha=%s catalogo=%s",
            var_fecha_proceso, var_catalogo)

## 4. Seccion constantes
> **Descripcion:** Valores que se mantienen constantes a lo largo del proceso.

In [ ]:
TBL_SALDO_ORIGEN = f"{var_catalogo}.fin.h_saldo"
TBL_SALDO_FINAL = f"{var_catalogo}.finanzas.fct_saldo_cierre"

TIP_MONEDA_LOCAL = "PEN"

## 5. Funciones de transformacion
> **Descripcion:** Funciones modularizadas de lectura, transformacion y escritura.

In [ ]:
def read_saldo_periodo(tabla, periodo):
    """Lee los saldos del periodo proyectando solo lo necesario."""
    return (
        spark.table(tabla)
        .select("cod_producto", "cod_cliente", "mto_saldo", "tip_moneda", "fec_periodo")
        .filter(F.col("fec_periodo") == periodo)
    )


def calculate_saldo_local(df_origen):
    """Expresa el saldo en moneda local aplicando el tipo de cambio vigente."""
    return df_origen.withColumn(
        "mto_saldo_local",
        F.when(F.col("tip_moneda") == TIP_MONEDA_LOCAL, F.col("mto_saldo"))
        .otherwise(F.col("mto_saldo") * F.col("val_tipo_cambio"))
    )


def group_saldo_producto(df_origen):
    """Agrupa el saldo por producto."""
    return (
        df_origen
        .groupBy("cod_producto", "fec_periodo")
        .agg(F.sum("mto_saldo_local").alias("mto_saldo_total"))
    )

## 6. Logica del proceso
> **Descripcion:** Orquestacion de las funciones definidas previamente.

In [ ]:
ini_etapa = time.perf_counter()

df_saldo = read_saldo_periodo(TBL_SALDO_ORIGEN, var_fecha_proceso)
df_saldo_local = calculate_saldo_local(df_saldo)
df_saldo_producto = group_saldo_producto(df_saldo_local)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 7. Deduplicacion
> **Descripcion:** Logica de deduplicacion segun las llaves de la tabla.

In [ ]:
ventana_producto = Window.partitionBy("cod_producto", "fec_periodo").orderBy(
    F.col("mto_saldo_total").desc()
)

df_saldo_unico = (
    df_saldo_producto
    .withColumn("nro_orden", F.row_number().over(ventana_producto))
    .filter(F.col("nro_orden") == 1)
    .drop("nro_orden")
)

## 8. Escritura en la tabla final
> **Descripcion:** Persistencia del resultado en formato Delta.

In [ ]:
ini_escritura = time.perf_counter()

try:
    (
        df_saldo_unico
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TBL_SALDO_FINAL)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_SALDO_FINAL, exc)
    raise

logger.info("Tiempo de escritura: %.2f segundos",
            time.perf_counter() - ini_escritura)

## 9. Registro de la ejecucion
> **Descripcion:** Cierre del proceso con el registro de duracion y volumen.

In [ ]:
logger.info("Fin del proceso ETL_CIERRE_SALDOS. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)